# 00 · Setup and inspect the raw sources

**Questions**
1. Did every download arrive, and how much of it?
2. What are the real variable names, units and sampling rates? (`docs/DATASETS.md` had them as "not confirmed".)
3. Is anything missing that changes the plan?

**Inputs:** `data/raw/pmdata/`, `data/raw/msl_rad/`, `data/raw/osdr_eda/` (all real data, nothing simulated here).
**Outputs:** the confirmed facts, written into `docs/DATASETS.md`. No processed tables yet.

In [1]:
import sys
sys.path.insert(0, "../ml")
import json
import cdflib
import numpy as np
import pandas as pd
from sol_ml.nbutil import *
from sol_ml.data import pmdata

rng = setup()
pd.set_option("display.width", 200)

## 1. PMData (the astronaut's body)

In [2]:
people = pmdata.participants()
rows = []
for p in people:
    w = pmdata.load_wellness(p)
    rhr = pmdata.load_resting_hr(p).dropna()
    score = pmdata.load_sleep_score(p)
    rows.append({"person": p.name, "first_day": w.index.min().date(), "last_day": w.index.max().date(),
                 "wellness_days": len(w), "resting_hr_days": len(rhr), "sleep_score_nights": len(score),
                 "injury_file": (p / "pmsys" / "injury.csv").exists(),
                 "heart_rate_mb": round((p / "fitbit" / "heart_rate.json").stat().st_size / 1e6)})
pm = pd.DataFrame(rows).set_index("person")
print(f"{len(people)} participants, {pm.first_day.min()} to {pm.last_day.max()}")
pm

16 participants, 2019-11-01 to 2020-03-30


,first_day,last_day,wellness_days,resting_hr_days,sleep_score_nights,injury_file,heart_rate_mb
person,,,,,,,
p01,2019-11-01,2020-03-30,138,152,150,True,120
p02,2019-11-03,2020-03-30,99,89,138,True,112
p03,2019-11-01,2020-03-30,78,101,73,True,61
p04,2019-11-01,2020-03-30,144,35,139,True,120
p05,2019-11-01,2020-03-30,134,86,117,True,104
p06,2019-11-02,2020-03-30,147,150,146,True,120
p07,2019-11-06,2020-03-29,132,144,140,True,120
p08,2019-11-11,2020-03-27,103,139,132,False,123
p09,2019-11-15,2020-03-30,102,147,131,True,99


In [3]:
print((pmdata.RAW / "license.txt").read_text().split("\n")[0])
pd.read_csv(people[0] / "pmsys" / "wellness.csv").head(3)

The license for the PMData dataset is Attribution-NonCommercial 4.0 International.


,effective_time_frame,fatigue,mood,readiness,sleep_duration_h,sleep_quality,soreness,soreness_area,stress
0,2019-11-01T08:31:40.751Z,2,3,5,6,3,2,[12921003],3
1,2019-11-02T10:00:01.229Z,2,3,6,6,3,2,[12921003],3
2,2019-11-03T14:28:03.263Z,3,3,8,6,3,3,[],3


## 2. MSL RAD (radiation on the Mars surface)

In [4]:
cdf_files = sorted((RAW / "msl_rad").rglob("*.cdf"))
print(len(cdf_files), "daily CDF files:", {y.name: len(list(y.glob("*.cdf"))) for y in sorted((RAW / "msl_rad").iterdir())})
cdf = cdflib.CDF(str(cdf_files[0]))
pd.DataFrame([{"variable": v, "dims": cdf.varinq(v).Dim_Sizes, "units": cdf.varattsget(v).get("UNITS", ""),
               "description": str(cdf.varattsget(v).get("CATDESC", ""))[:70]} for v in cdf.cdf_info().zVariables])

388 daily CDF files: {'2024': 287, '2026': 101}


,variable,dims,units,description
0,ALIVE_TIME,[],seconds,Actual instrument observation time
1,DOSIMETRY_TOTAL,[3],uGray/day,"Radiation dose rate on Mars (recorded in Si, c..."
2,DOSIMETRY_TIMESLICE,"[16, 3]",uGray/day,Total Dose for an observation at 1/16th timesl...
3,DOSIMETRY_TIMESLICE_AXIS,[16],rclk,Bin start time
4,DOSIMETRY_DOSE_AXIS,[3],uGray/day,Total Dose
5,LET_A1A2,[52],counts,LET spectrum on Mars in Si (geometric factor ~...
6,LET_BIN_ENERGY,[52],keV/micron,Energy at the start of the bin
7,HISTOGRAM_NEUTRAL_DE,[96],counts,Neutral particle (neutrons + gamma rays) count...
8,HISTOGRAM_NEUTRAL_D,[96],counts,Neutral particle counts in CsI
9,HISTOGRAM_NEUTRAL_E,[96],counts,Neutral particle counts in tissue-equivalent p...


In [5]:
labels = [str(x) for x in np.ravel(cdf.varget("DOSIMETRY_DOSE_LABEL"))[:3]]
t = pd.to_datetime(cdflib.cdfepoch.to_datetime(cdf.varget("EPOCH")))
dose = pd.DataFrame(cdf.varget("DOSIMETRY_TOTAL"), columns=labels, index=t)
print("dose columns:", labels)
print("observations in", cdf_files[0].name, ":", len(t), "| median step:", t.to_series().diff().median())
dose.describe().loc[["50%", "max"]].round(1)

dose columns: ['Total Dose B', 'RTG Corrected Dose B', 'Total Dose E']
observations in msl_rad_obs-l2_2024001_v04.cdf : 85 | median step: 0 days 00:17:06


,Total Dose B,RTG Corrected Dose B,Total Dose E
50%,190.6,131.7,180.2
max,237.7,178.9,195.3


## 3. OSDR EDA (ISS cabin air and ISS radiation)

In [6]:
rows = []
for f in sorted((RAW / "osdr_eda").glob("*_telemetry.csv")):
    df = pd.read_csv(f, parse_dates=["time"])
    name = f.name.replace("_telemetry.csv", "")
    rad = RAW / "osdr_eda" / f"{name}_radiation.csv"
    co2 = df["co2_iss"].dropna()
    rows.append({"mission": name, "start": df.time.min().date(), "days": round((df.time.max() - df.time.min()).days),
                 "median_step": str(df.time.sort_values().diff().median()).replace("0 days ", ""),
                 "co2_iss_rows": len(co2), "co2_median_ppm": co2.median(),
                 "co2_median_mmHg": round(co2.median() * 760 / 1e6, 2) if len(co2) else None,
                 "radiation_days": len(pd.read_csv(rad)) if rad.exists() else 0})
eda = pd.DataFrame(rows).set_index("mission").sort_values("days", ascending=False)
eda

,start,days,median_step,co2_iss_rows,co2_median_ppm,co2_median_mmHg,radiation_days
mission,,,,,,,
RR-18,2021-12-20,242,00:01:00,0,NaN,NaN,0
RR-20,2023-11-10,172,00:01:00,0,NaN,NaN,0
RR-7,2018-06-29,81,00:00:00,21948,3368.0,2.56,76
RR-5,2017-06-01,68,00:05:00,19160,2422.0,1.84,0
RR-25,2022-11-29,65,00:01:00,0,NaN,NaN,0
RR-6,2017-12-06,61,00:05:00,17572,3151.0,2.39,49
RR-3,2016-04-08,49,00:05:00,14385,3638.0,2.76,43
RR-28,2024-03-21,46,00:01:00,0,NaN,NaN,0
RR-1,2014-09-20,41,00:05:00,12087,3192.0,2.43,39


In [7]:
pd.read_csv(RAW / "osdr_eda" / "RR-1_radiation.csv").head(3)

,time,gcr,saa,total,milestones
0,2014-09-21T00:00:00Z,0.13395,0.07158,0.20553,Launch
1,2014-09-22T00:00:00Z,0.13334,0.09724,0.23058,NaN
2,2014-09-23T00:00:00Z,0.13589,0.06360,0.19949,NaN


## Decision

Nothing is modelled here; this notebook fixes the facts the other notebooks build on. All of them are now in `docs/DATASETS.md`.

**Confirmed**
- **PMData:** 16 participants, 1 Nov 2019 to 30 Mar 2020. Mirror license is CC BY-NC 4.0.
- **MSL RAD:** dose rate is `DOSIMETRY_TOTAL` (µGy/day) with three columns: *Total Dose B* (silicon, includes the rover's RTG),
  *RTG Corrected Dose B*, and *Total Dose E* (tissue-equivalent plastic). Time is `EPOCH`. Observations are irregular, roughly every 17 minutes.
- **OSDR EDA:** CO2 is in **ppm** (not mmHg or %), so `mmHg = ppm × 760 / 1e6`. Sampling is every 5 minutes for the older missions
  and every minute for the newer ones. Radiation is daily in mGy/day with `gcr`, `saa` and `total`.

**What does not match the plan (said here and in chat, per the team rule)**
1. **Two people have no usable heart data.** p12 has one night of Fitbit data and p13 has 47. The body models will run on about 14 people, not 16.
2. **The longest ISS missions have no CO2.** RR-18 (243 days), RR-20, RR-25 and RR-28 carry temperature and humidity only.
   CO2 work uses RR-7, RR-5, RR-6, RR-3, RR-1, RR-12, RR-10, RR-23, RR-19 and RR-9 (35–82 days each).
3. **Median ISS CO2 already sits near the team's limits** (1.6–2.8 mmHg per mission against limits of 2.5 and 3 mmHg).
   The environmental gate will be active for long stretches of the real series; notebook 07 must quantify that before the thresholds are final.
4. **OSDR has no accumulated-dose column.** The dose counter sums `total` itself.
5. OSDR EDA was fetched through its public API (`data/fetch/fetch_osdr_eda.py`) instead of by hand. The tables are the same.

**Known limits:** PMData people are healthy athletes on Earth, so it has no medical events and no BP, SpO2, temperature or glucose.